# 11. Safe SQL Generation

**Practical use case:** Generate and validate read-only analytical SQL.

This trainer-ready notebook contains explanation, live `langchain_openai` code, validation guidance and exercises. It makes real API calls and may incur charges.

## 1. Business problem

**Scenario:** Generate read-only analytical SQL from a business question.

The goal is to convert an unstructured language task into a repeatable workflow that can be demonstrated, tested and later integrated into an application.

## 2. Solution workflow

1. Prepare or load the input.
2. Define the model and important parameters.
3. Construct a precise prompt or schema.
4. Invoke the model.
5. inspect and validate the response.
6. Save or pass the result to the next application step.

### 1. Set up the API key and model

This cell imports the required classes, reads the API key securely, and selects the model without exposing credentials.

**Expected result:** No model output is produced; the environment becomes ready for later API calls. Read the output before continuing to the next cell.

In [ ]:
import os, getpass
from langchain_openai import ChatOpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY: ")

MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

### 2. Validate the generated result

This cell checks important business and safety rules before the generated result is accepted.

**Expected result:** Boolean checks or assertions confirm whether the result is valid. Read the output before continuing to the next cell.

In [ ]:
schema = "orders(order_id INTEGER, order_date DATE, region TEXT, amount DECIMAL, status TEXT)"
question = "Show total completed-order revenue by region for 2026, highest first."
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
prompt = f"""Generate one PostgreSQL SELECT query only.
Schema: {schema}
Question: {question}
Rules: SELECT only; no INSERT, UPDATE, DELETE, DROP, ALTER, or comments."""
sql = llm.invoke(prompt).content.strip().replace("```sql", "").replace("```", "").strip()
blocked = ["insert", "update", "delete", "drop", "alter", "truncate"]
is_safe = sql.lower().startswith("select") and not any(word in sql.lower() for word in blocked)
print(sql)
print("Passed basic safety check:", is_safe)
# Do not execute generated SQL without parsing, authorization and database controls.

## Expected result

The model should return an output that follows the requested scope and format. During training, compare the actual response with the requirement instead of assuming that a fluent response is correct.

## Parameter experiment

Run the example once with `temperature=0` and once with a higher supported temperature. Compare consistency, wording and creativity. Keep other inputs unchanged so the comparison is meaningful.

## Output validation

Check that the response:

- follows the requested format;
- contains no invented facts;
- preserves important names and numbers;
- is relevant to the business question;
- can be safely used by the next system.

## Error handling

Production code should handle missing API keys, authentication failures, rate limits, timeouts, invalid structured output and temporary service errors. Use limited retries and provide a clear fallback message.

### 3. Call the model and inspect the response

This cell calls the configured model with the prepared prompt and extracts the returned content.

**Expected result:** A live model response is printed; wording can vary unless deterministic settings are used. Read the output before continuing to the next cell.

In [ ]:
question2="Count orders by status for the previous month."
sql2=llm.invoke(f"Generate one PostgreSQL SELECT query only. Schema: {schema}. Question: {question2}").content
print(sql2)